# 03 — Load a Matchup Dataset

Demonstrates downloading source products and loading a matchup as an `xr.DataTree`.

**What happens:**
1. Open the catalogue from `01_find_and_catalogue.ipynb`.
2. Download the source products (Sentinel-2A and Landsat-9 scenes) if not already present.
3. Load the collocated dataset via `Matchup.return_matchup_dataset()`.
4. Explore the resulting `xr.DataTree` structure and plot a quick RGB preview.

**Prerequisites:** run `01_find_and_catalogue.ipynb` first.

> **Note:** downloading large EO products can take several minutes depending on your connection.

In [ ]:
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

from eomatch import MatchupCatalogue

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CATALOGUE_DIR = os.path.join(NOTEBOOK_DIR, "catalogue")

cat = MatchupCatalogue.open(CATALOGUE_DIR)
print("Opened:", cat.path)

## Download source products

`MatchupCatalogue.download_products` fetches each product from the public archives (CEDA / AWS)
and registers a `"data"` asset on the corresponding STAC item. Products that are already on disk
are registered without re-downloading.

In [ ]:
downloaded = cat.download_products()
print(f"Products available ({len(downloaded)}):")
for path in downloaded:
    print(f"  {path}")

## Select an event and matchup

Filter to events whose products are already downloaded before attempting to load data.

In [ ]:
events = cat.get_events(products_downloaded=True)
print(f"Events with all products downloaded: {len(events)}")

if not events:
    raise RuntimeError("No downloaded events found — run the download cell above first.")

event = events[0]
mu = event.matchup_set[0]

print(f"\nEvent:   {event.stac_id}")
print(f"Matchup: {mu.stac_id}")
print(f"Products:")
for p in mu.products:
    print(f"  [{p.collection}]  {p.id}")

## Load the DataTree

`Matchup.return_matchup_dataset()` reads each source product with `eoio` and assembles a
`xr.DataTree` with one leaf node per sensor (`sensor_1`, `sensor_2`, …).

Pass `collection_read_args` to select specific bands per collection — the keys are STAC
collection IDs and the values override the default read parameters.

In [ ]:
dt_mu = mu.return_matchup_dataset(
    collection_read_args={
        "S2_MSI_L1C":   {"vars_sel": {"meas": ["B02", "B03", "B04", "B08"]}},
        "LANDSAT_C2L1": {"vars_sel": {"meas": ["B2",  "B3",  "B4",  "B5" ]}},
    }
)

print(dt_mu)

## Explore the tree structure

In [ ]:
for name, node in dt_mu.subtree:
    if not node.ds:
        continue
    ds = node.ds
    print(f"/{name}")
    print(f"  dims:  {dict(ds.dims)}")
    print(f"  vars:  {list(ds.data_vars)}")
    print(f"  CRS:   {ds.rio.crs if hasattr(ds, 'rio') else 'n/a'}")

## Plot an RGB preview

Quick false-colour preview using the red, green, and blue bands for each sensor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BAND_MAP = {
    "S2_MSI_L1C":   {"r": "B04", "g": "B03", "b": "B02"},
    "LANDSAT_C2L1": {"r": "B4",  "g": "B3",  "b": "B2"},
}

nodes = [(name, node) for name, node in dt_mu.subtree if node.ds and node.ds.data_vars]
fig, axes = plt.subplots(1, len(nodes), figsize=(6 * len(nodes), 5))
if len(nodes) == 1:
    axes = [axes]

for ax, (name, node) in zip(axes, nodes):
    ds = node.ds
    # Identify collection from a coordinate or attribute if present
    collection = ds.attrs.get("collection", "")
    bmap = BAND_MAP.get(collection, {})
    if not bmap:
        # Fall back: use first three measurement variables
        bands = list(ds.data_vars)[:3]
        r, g, b = (ds[bands[i]].values for i in range(min(3, len(bands))))
    else:
        r = ds[bmap["r"]].values if bmap["r"] in ds else ds[list(ds.data_vars)[0]].values
        g = ds[bmap["g"]].values if bmap["g"] in ds else r
        b = ds[bmap["b"]].values if bmap["b"] in ds else r

    def _normalise(arr):
        arr = np.squeeze(arr).astype(float)
        p2, p98 = np.nanpercentile(arr, [2, 98])
        return np.clip((arr - p2) / (p98 - p2 + 1e-9), 0, 1)

    rgb = np.stack([_normalise(r), _normalise(g), _normalise(b)], axis=-1)
    ax.imshow(rgb)
    ax.set_title(f"{name}\n{collection or ''}")
    ax.axis("off")

plt.suptitle(f"Matchup preview — {mu.stac_id[:60]}", fontsize=9)
plt.tight_layout()
plt.show()